# 🔮 Inference & Local Explainability - HarmonyMind
This notebook runs local inference predictions on arbitrary input sentences and uses **LIME-style attributions** to explain the model's confidence in its predictions.


In [ ]:
import os
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences


## Load Saved Model Assets


In [ ]:
project_root = os.path.dirname(os.getcwd())
with open(os.path.join(project_root, 'models', 'tokenizer.pkl'), 'rb') as f:
    tokenizer = pickle.load(f)
with open(os.path.join(project_root, 'models', 'label_encoder.pkl'), 'rb') as f:
    le = pickle.load(f)

model = tf.keras.models.load_model(os.path.join(project_root, 'models', 'best_model.h5'))
MAX_LEN = 100


## Define Prediction and LIME Explanation Functions


In [ ]:
def predict_sentence(text):
    seq = pad_sequences(tokenizer.texts_to_sequences([text]), maxlen=MAX_LEN)
    pred = model.predict(seq, verbose=0)[0]
    emotion = le.classes_[np.argmax(pred)]
    confidence = np.max(pred)
    return emotion, confidence, pred

def explain_lime(text, target_emotion):
    words = text.split()
    if len(words) == 0: return []
    
    _, _, pred = predict_sentence(text)
    class_idx = list(le.classes_).index(target_emotion)
    orig_conf = pred[class_idx]
    
    attributions = []
    for i in range(len(words)):
        perturbed = words[:i] + words[i+1:]
        perturbed_text = ' '.join(perturbed)
        
        if len(perturbed) == 0:
            pert_conf = 0.0
        else:
            _, _, pert_pred = predict_sentence(perturbed_text)
            pert_conf = pert_pred[class_idx]
            
        score = orig_conf - pert_conf
        attributions.append((words[i], score))
    return attributions


## Run Sample Test Predictions


In [ ]:
sample_text = "I am feeling incredibly overwhelmed and anxious about this project deadline"
emo, conf, _ = predict_sentence(sample_text)
print(f'Text: "{sample_text}"')
print(f'Predicted Emotion: {emo.upper()} (Confidence: {conf*100:.1f}%)')

print('\nLIME Word Attributions:')
attributions = explain_lime(sample_text, emo)
for word, score in attributions:
    print(f'  {word:<15} : {score:+.4f}')
